# Phase 13 — Target Alignment & Leakage Correction

**Project:** Australian Stock Market Analytics & Return Modeling

## Purpose
This phase audits how the 3-month forward-return target was constructed and creates a stricter, auditable target table with explicit future endpoint dates. It also creates a conservative predictor allowlist and a chronological evaluation split that excludes observations whose target endpoint crosses a split boundary.

**Important:** This notebook does not assume that the existing Phase 9 labels are correct. It attempts to reconstruct the target from Phase 8 daily prices and compares it with Phase 9 labels. Review discrepancies before using the corrected labels for modeling.

## 1. Setup and locate project files

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PRICE_REL = Path("data/processed/market_data/asx_company_daily_prices.csv")
FEATURE_REL = Path("data/processed/model_features/asx_3m_return_features_and_targets.csv")
PHASE9_REL = Path("data/processed/model_targets/asx_forward_return_targets_all_rows.csv")

def locate_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / PRICE_REL).exists() and (p / FEATURE_REL).exists():
            return p
    for p in start.iterdir() if start.exists() else []:
        if p.is_dir() and (p / PRICE_REL).exists() and (p / FEATURE_REL).exists():
            return p
    return None

PROJECT_ROOT = locate_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find both Phase 8 prices and Phase 10 feature table. "
        "Set PROJECT_ROOT manually to your project root."
    )

PRICE_PATH = PROJECT_ROOT / PRICE_REL
FEATURE_PATH = PROJECT_ROOT / FEATURE_REL
PHASE9_PATH = PROJECT_ROOT / PHASE9_REL
OUT_DIR = PROJECT_ROOT / "reports" / "target_alignment_audit"
TARGET_DIR = PROJECT_ROOT / "data" / "processed" / "model_targets"
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Phase 8 prices:", PRICE_PATH.exists())
print("Phase 9 target table:", PHASE9_PATH.exists())
print("Phase 10 feature table:", FEATURE_PATH.exists())

Project root: e:\Projects\Australian Stock Market Data Science Project
Phase 8 prices: True
Phase 9 target table: True
Phase 10 feature table: True


## 2. Load Phase 8 prices and inspect schema

We need ticker, date, and a price series. Prefer `adj_close` if it exists; otherwise use `close`. This choice is recorded in the audit.

In [2]:
prices = pd.read_csv(PRICE_PATH, low_memory=False)
prices.columns = [str(c).strip().lower() for c in prices.columns]

# Normalize common aliases without inventing data.
rename_map = {}
if "ticker" not in prices.columns:
    for candidate in ["symbol", "stock", "company_ticker"]:
        if candidate in prices.columns:
            rename_map[candidate] = "ticker"
            break
if "date" not in prices.columns:
    for candidate in ["datetime", "timestamp", "trading_date"]:
        if candidate in prices.columns:
            rename_map[candidate] = "date"
            break
prices = prices.rename(columns=rename_map)

if not {"ticker", "date"}.issubset(prices.columns):
    raise ValueError(f"Phase 8 price file must contain ticker and date. Found: {prices.columns.tolist()}")

price_col = "adj_close" if "adj_close" in prices.columns else ("close" if "close" in prices.columns else None)
if price_col is None:
    raise ValueError(f"No adj_close or close column found. Columns: {prices.columns.tolist()}")

prices["ticker"] = prices["ticker"].astype(str).str.strip().str.upper()
prices["date"] = pd.to_datetime(prices["date"], errors="coerce")
prices[price_col] = pd.to_numeric(prices[price_col], errors="coerce")
prices = prices.dropna(subset=["ticker", "date", price_col])
prices = prices[prices[price_col] > 0].copy()
prices = prices.sort_values(["ticker", "date"]).drop_duplicates(["ticker", "date"], keep="last")
prices = prices.reset_index(drop=True)

print("Rows:", len(prices))
print("Tickers:", prices["ticker"].nunique())
print("Date range:", prices["date"].min(), "to", prices["date"].max())
print("Price column used:", price_col)
display(prices.head())

Rows: 65771
Tickers: 52
Date range: 2021-10-19 00:00:00 to 2026-10-19 00:00:00
Price column used: close


,date,close,high,low,open,volume,yahoo_symbol,ticker
0,2021-10-19,6.458141,6.467407,6.143110,6.143110,5232326,A2M.AX,A2M
1,2021-10-20,6.847298,6.902892,6.485939,6.532267,15237359,A2M.AX,A2M
2,2021-10-21,6.606391,6.847297,6.569328,6.847297,6015791,A2M.AX,A2M
3,2021-10-22,6.384016,6.597125,6.346954,6.597125,6359080,A2M.AX,A2M
4,2021-10-25,6.300626,6.439610,6.272829,6.393282,4083248,A2M.AX,A2M


## 3. Reconstruct 3-month forward returns and endpoint dates

The horizon is 63 future observations **within each ticker**, using the price series available in Phase 8. This is an observation-based horizon, not exactly three calendar months. Each target records the date and price used as its endpoint.

Formula: `forward_return_3m = endpoint_price / current_price - 1`.

Rows without a 63-observation future endpoint remain unlabeled.

In [3]:
prices["close_for_target"] = prices[price_col].astype(float)
grouped = prices.groupby("ticker", sort=False)
prices["target_endpoint_date_3m"] = grouped["date"].shift(-63)
prices["target_endpoint_price_3m"] = grouped["close_for_target"].shift(-63)
prices["forward_return_3m_reconstructed"] = (
    prices["target_endpoint_price_3m"] / prices["close_for_target"] - 1
)
prices["target_horizon_observations"] = 63

reconstructed = prices[[
    "ticker", "date", "close_for_target",
    "target_endpoint_date_3m", "target_endpoint_price_3m",
    "forward_return_3m_reconstructed", "target_horizon_observations"
]].copy()

print("Reconstructed rows:", len(reconstructed))
print("Rows with valid 3-month target:", reconstructed["forward_return_3m_reconstructed"].notna().sum())
print("Tickers:", reconstructed["ticker"].nunique())
display(reconstructed.head())

Reconstructed rows: 65771
Rows with valid 3-month target: 62495
Tickers: 52


,ticker,date,close_for_target,target_endpoint_date_3m,target_endpoint_price_3m,forward_return_3m_reconstructed,target_horizon_observations
0,A2M,2021-10-19,6.458141,2022-01-19,4.975641,-0.229555,63
1,A2M,2021-10-20,6.847298,2022-01-20,4.947844,-0.277402,63
2,A2M,2021-10-21,6.606391,2022-01-21,4.744000,-0.281907,63
3,A2M,2021-10-22,6.384016,2022-01-24,4.716204,-0.261248,63
4,A2M,2021-10-25,6.300626,2022-01-25,5.049766,-0.198529,63


## 4. Compare reconstructed labels with Phase 9 and Phase 10 labels

This comparison helps identify whether the prior target used the same horizon, price field, or date alignment. Differences are not automatically errors: Phase 9 may have used different documented settings. Investigate large differences rather than silently overwriting labels.

In [4]:
features = pd.read_csv(FEATURE_PATH, low_memory=False)
features.columns = [str(c).strip().lower() for c in features.columns]
features["ticker"] = features["ticker"].astype(str).str.strip().str.upper()
features["date"] = pd.to_datetime(features["date"], errors="coerce")
if "forward_return_3m" not in features.columns:
    raise ValueError("Phase 10 feature table does not contain forward_return_3m.")
features["forward_return_3m"] = pd.to_numeric(features["forward_return_3m"], errors="coerce")

compare = reconstructed.merge(
    features[["ticker", "date", "forward_return_3m"]],
    on=["ticker", "date"], how="left", validate="one_to_one"
)
compare["difference_reconstructed_minus_phase10"] = (
    compare["forward_return_3m_reconstructed"] - compare["forward_return_3m"]
)
compare["absolute_difference"] = compare["difference_reconstructed_minus_phase10"].abs()

comparison_summary = pd.DataFrame([{
    "rows_matched": int(compare["forward_return_3m"].notna().sum()),
    "reconstructed_target_rows": int(compare["forward_return_3m_reconstructed"].notna().sum()),
    "phase10_target_rows": int(compare["forward_return_3m"].notna().sum()),
    "rows_with_both_targets": int(compare[["forward_return_3m_reconstructed", "forward_return_3m"]].notna().all(axis=1).sum()),
    "mean_absolute_difference": float(compare["absolute_difference"].mean()),
    "median_absolute_difference": float(compare["absolute_difference"].median()),
    "correlation": float(compare[["forward_return_3m_reconstructed", "forward_return_3m"]].corr().iloc[0, 1])
        if compare[["forward_return_3m_reconstructed", "forward_return_3m"]].dropna().shape[0] > 1 else np.nan
}])
display(comparison_summary)
comparison_summary.to_csv(OUT_DIR / "target_reconstruction_comparison_summary.csv", index=False)
compare.to_csv(OUT_DIR / "target_reconstruction_row_comparison.csv", index=False)

if PHASE9_PATH.exists():
    phase9 = pd.read_csv(PHASE9_PATH, low_memory=False)
    phase9.columns = [str(c).strip().lower() for c in phase9.columns]
    print("Phase 9 target columns:", phase9.columns.tolist())
else:
    phase9 = None
    print("Phase 9 all-rows target file not found; compared reconstructed labels against Phase 10 labels only.")

,rows_matched,reconstructed_target_rows,phase10_target_rows,rows_with_both_targets,mean_absolute_difference,median_absolute_difference,correlation
0,62495,62495,62495,62495,4.030331e-17,4.163336e-17,1.0


Phase 9 target columns: ['date', 'close', 'high', 'low', 'open', 'volume', 'yahoo_symbol', 'ticker', 'adj_close', 'price_for_returns', 'future_price_1m', 'forward_return_1m', 'future_price_3m', 'forward_return_3m', 'future_price_6m', 'forward_return_6m', 'benchmark_forward_return_1m', 'benchmark_forward_return_3m', 'benchmark_forward_return_6m', 'relative_forward_return_1m', 'outperforms_benchmark_1m', 'relative_forward_return_3m', 'outperforms_benchmark_3m', 'relative_forward_return_6m', 'outperforms_benchmark_6m']


## 5. Create corrected target table with explicit endpoint

The corrected table is based on Phase 8 price observations and retains the exact endpoint date and endpoint price used for the target. It is saved separately so the original Phase 9 outputs remain unchanged.

In [5]:
corrected_targets = reconstructed.rename(columns={
    "forward_return_3m_reconstructed": "forward_return_3m"
}).copy()

corrected_targets["target_source_price_column"] = price_col
corrected_targets["target_definition"] = "price at t+63 ticker observations divided by price at t, minus 1"
corrected_targets.to_csv(TARGET_DIR / "asx_forward_return_3m_targets_with_endpoints.csv", index=False)

print("Saved corrected target table:", TARGET_DIR / "asx_forward_return_3m_targets_with_endpoints.csv")
display(corrected_targets.head())

Saved corrected target table: e:\Projects\Australian Stock Market Data Science Project\data\processed\model_targets\asx_forward_return_3m_targets_with_endpoints.csv


,ticker,date,close_for_target,target_endpoint_date_3m,target_endpoint_price_3m,forward_return_3m,target_horizon_observations,target_source_price_column,target_definition
0,A2M,2021-10-19,6.458141,2022-01-19,4.975641,-0.229555,63,close,price at t+63 ticker observations divided by p...
1,A2M,2021-10-20,6.847298,2022-01-20,4.947844,-0.277402,63,close,price at t+63 ticker observations divided by p...
2,A2M,2021-10-21,6.606391,2022-01-21,4.744000,-0.281907,63,close,price at t+63 ticker observations divided by p...
3,A2M,2021-10-22,6.384016,2022-01-24,4.716204,-0.261248,63,close,price at t+63 ticker observations divided by p...
4,A2M,2021-10-25,6.300626,2022-01-25,5.049766,-0.198529,63,close,price at t+63 ticker observations divided by p...


## 6. Create a conservative predictor allowlist

A name-based exclusion is not enough to guarantee no leakage. For this first corrected model table, the safest approach is to allow only features with a clear point-in-time interpretation. The default list below is intentionally conservative and includes lagged/rolling market features when present. Manually review each feature's construction before modeling.

Features not listed are excluded by default.

In [6]:
candidate_safe_features = [
    "return_1d", "return_5d", "return_21d", "return_63d",
    "rolling_volatility_21d", "rolling_volatility_63d",
    "sma_21", "sma_63", "price_to_sma_21", "price_to_sma_63",
    "momentum_21d", "momentum_63d",
    "benchmark_return_1d", "benchmark_return_21d", "benchmark_return_63d",
    "benchmark_volatility_21d", "benchmark_volatility_63d",
    "relative_momentum_21d", "relative_momentum_63d"
]
available_safe_features = [c for c in candidate_safe_features if c in features.columns]

print("Safe candidate features available in Phase 10 table:")
for c in available_safe_features:
    print(" -", c)
if not available_safe_features:
    print("No allowlisted names matched. Inspect the Phase 10 columns and update the allowlist manually.")

feature_review = pd.DataFrame({
    "column": features.columns,
    "allowlisted_for_corrected_model": [c in available_safe_features for c in features.columns],
    "note": [
        "Candidate only; verify feature uses data available at or before the row date."
        if c in available_safe_features else
        "Excluded by conservative allowlist; manually review before adding."
        for c in features.columns
    ]
})
feature_review.to_csv(OUT_DIR / "conservative_feature_allowlist_review.csv", index=False)
display(feature_review)

Safe candidate features available in Phase 10 table:
 - momentum_21d
 - momentum_63d
 - benchmark_return_1d
 - benchmark_volatility_21d
 - benchmark_volatility_63d


,column,allowlisted_for_corrected_model,note
0,ticker,False,Excluded by conservative allowlist; manually r...
1,date,False,Excluded by conservative allowlist; manually r...
2,time_split,False,Excluded by conservative allowlist; manually r...
3,lagged_return_1d,False,Excluded by conservative allowlist; manually r...
4,lagged_return_5d,False,Excluded by conservative allowlist; manually r...
5,lagged_return_21d,False,Excluded by conservative allowlist; manually r...
6,lagged_return_63d,False,Excluded by conservative allowlist; manually r...
7,lagged_return_126d,False,Excluded by conservative allowlist; manually r...
8,volatility_5d,False,Excluded by conservative allowlist; manually r...
9,volatility_21d,False,Excluded by conservative allowlist; manually r...


## 7. Merge corrected targets with Phase 10 predictors

The merge uses `ticker` and `date`. The corrected target table is authoritative for the target in this new dataset; the original target remains available for comparison.

In [7]:
predictor_cols = ["ticker", "date"] + available_safe_features
predictor_table = features[predictor_cols].copy()
corrected_model_table = corrected_targets.merge(
    predictor_table, on=["ticker", "date"], how="left", validate="one_to_one"
)
corrected_model_table = corrected_model_table.sort_values(["date", "ticker"]).reset_index(drop=True)
corrected_model_table.to_csv(
    PROJECT_ROOT / "data" / "processed" / "model_features" / "asx_3m_corrected_model_table.csv",
    index=False
)
print("Corrected model table shape:", corrected_model_table.shape)
print("Target rows:", corrected_model_table["forward_return_3m"].notna().sum())
print("Safe features:", len(available_safe_features))

Corrected model table shape: (65771, 14)
Target rows: 62495
Safe features: 5


## 8. Build a strict endpoint-based chronological split audit

The split uses dates to define train/validation/test eras. For training rows, we require each target endpoint to be strictly before the validation period starts. For validation rows, we require each endpoint to be strictly before the test period starts. This is stricter and more transparent than dropping a fixed number of global dates.

In [8]:
labeled = corrected_model_table.dropna(subset=["forward_return_3m", "target_endpoint_date_3m"]).copy()
labeled["target_endpoint_date_3m"] = pd.to_datetime(labeled["target_endpoint_date_3m"], errors="coerce")
labeled = labeled.dropna(subset=["target_endpoint_date_3m"])
unique_dates = np.array(sorted(labeled["date"].unique()))
if len(unique_dates) < 20:
    raise ValueError("Too few dates to form chronological splits.")

cut1 = pd.Timestamp(unique_dates[max(1, int(len(unique_dates) * 0.70) - 1)])
cut2 = pd.Timestamp(unique_dates[max(2, int(len(unique_dates) * 0.85) - 1)])

train_start = labeled[labeled["date"] <= cut1].copy()
valid_period = labeled[(labeled["date"] > cut1) & (labeled["date"] <= cut2)].copy()
test_period = labeled[labeled["date"] > cut2].copy()

valid_start_date = valid_period["date"].min()
test_start_date = test_period["date"].min()

train_strict = train_start[train_start["target_endpoint_date_3m"] < valid_start_date].copy()
valid_strict = valid_period[valid_period["target_endpoint_date_3m"] < test_start_date].copy()
test_strict = test_period.copy()

split_audit = pd.DataFrame([
    {"split": "train_strict", "rows": len(train_strict), "tickers": train_strict["ticker"].nunique(),
     "first_date": train_strict["date"].min(), "last_date": train_strict["date"].max(),
     "latest_target_endpoint": train_strict["target_endpoint_date_3m"].max()},
    {"split": "validation_strict", "rows": len(valid_strict), "tickers": valid_strict["ticker"].nunique(),
     "first_date": valid_strict["date"].min(), "last_date": valid_strict["date"].max(),
     "latest_target_endpoint": valid_strict["target_endpoint_date_3m"].max()},
    {"split": "test", "rows": len(test_strict), "tickers": test_strict["ticker"].nunique(),
     "first_date": test_strict["date"].min(), "last_date": test_strict["date"].max(),
     "latest_target_endpoint": test_strict["target_endpoint_date_3m"].max()}
])
display(split_audit)
split_audit.to_csv(OUT_DIR / "strict_endpoint_split_audit.csv", index=False)

strict_train_path = PROJECT_ROOT / "data" / "processed" / "model_features" / "asx_3m_strict_split_train.csv"
strict_valid_path = PROJECT_ROOT / "data" / "processed" / "model_features" / "asx_3m_strict_split_validation.csv"
strict_test_path = PROJECT_ROOT / "data" / "processed" / "model_features" / "asx_3m_strict_split_test.csv"
train_strict.to_csv(strict_train_path, index=False)
valid_strict.to_csv(strict_valid_path, index=False)
test_strict.to_csv(strict_test_path, index=False)

,split,rows,tickers,first_date,last_date,latest_target_endpoint
0,train_strict,40447,52,2021-10-19,2024-11-14,2025-02-17
1,validation_strict,6084,52,2025-02-18,2025-08-05,2025-10-31
2,test,9412,52,2025-11-03,2026-07-22,2026-10-19


## 9. Summarize findings and next action

This notebook intentionally does not train another model. Its output determines whether Phases 10–12 should be rerun using corrected labels and stricter splits. Check the label comparison and split sizes before moving on.

In [9]:
summary = {
    "phase": 13,
    "price_column_used": price_col,
    "price_rows": int(len(prices)),
    "price_tickers": int(prices["ticker"].nunique()),
    "reconstructed_target_rows": int(corrected_targets["forward_return_3m"].notna().sum()),
    "phase10_target_rows": int(features["forward_return_3m"].notna().sum()),
    "candidate_safe_features": available_safe_features,
    "n_candidate_safe_features": int(len(available_safe_features)),
    "strict_split_rows": {
        "train": int(len(train_strict)),
        "validation": int(len(valid_strict)),
        "test": int(len(test_strict))
    },
    "outputs": {
        "corrected_targets": str(TARGET_DIR / "asx_forward_return_3m_targets_with_endpoints.csv"),
        "corrected_model_table": str(PROJECT_ROOT / "data" / "processed" / "model_features" / "asx_3m_corrected_model_table.csv"),
        "train": str(strict_train_path),
        "validation": str(strict_valid_path),
        "test": str(strict_test_path)
    },
    "warning": "The allowlist is based on column names and must be manually checked against feature calculation timing. Reconstructed targets must be compared with Phase 9 before replacing prior labels."
}
with open(OUT_DIR / "phase13_target_alignment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print("\nPhase 13 audit outputs saved to:", OUT_DIR)

{
  "phase": 13,
  "price_column_used": "close",
  "price_rows": 65771,
  "price_tickers": 52,
  "reconstructed_target_rows": 62495,
  "phase10_target_rows": 62495,
  "candidate_safe_features": [
    "momentum_21d",
    "momentum_63d",
    "benchmark_return_1d",
    "benchmark_volatility_21d",
    "benchmark_volatility_63d"
  ],
  "n_candidate_safe_features": 5,
  "strict_split_rows": {
    "train": 40447,
    "validation": 6084,
    "test": 9412
  },
  "outputs": {
    "corrected_targets": "e:\\Projects\\Australian Stock Market Data Science Project\\data\\processed\\model_targets\\asx_forward_return_3m_targets_with_endpoints.csv",
    "corrected_model_table": "e:\\Projects\\Australian Stock Market Data Science Project\\data\\processed\\model_features\\asx_3m_corrected_model_table.csv",
    "train": "e:\\Projects\\Australian Stock Market Data Science Project\\data\\processed\\model_features\\asx_3m_strict_split_train.csv",
    "validation": "e:\\Projects\\Australian Stock Market Data S

## 10. Completion checklist

Before marking this phase complete:
- Review the reconstructed-vs-Phase-10 target comparison.
- Verify the chosen price column and 63-observation horizon match the intended research definition.
- Review the conservative feature allowlist and feature engineering definitions.
- Confirm train, validation, and test have nonzero rows and the target endpoints do not cross into the next split.
- If the reconstructed targets differ substantially from Phase 9, do not continue with the old Phase 11 metrics; investigate and rerun downstream phases.